In [34]:
import tensorflow as tf
import numpy as np
# import tensorflow_datasets as tfds
from datasets import load_dataset, load_from_disk

In [22]:
import os
try:
    import tensorflow_datasets as tfds
except ImportError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'tensorflow_datasets'])
    import tensorflow_datasets as tfds

In [23]:
tf.keras.utils.set_random_seed(42)
tf.__version__
tf.executing_eagerly()

True

In [24]:
tiny_texts = [
    "great movie",
    "amazing story",
    "bad movie",
    "terrible story"
]
tiny_vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=20,
    output_mode='int',
    output_sequence_length=4,
)
tiny_vectorizer.adapt(tiny_texts)
tiny_vocab = tiny_vectorizer.get_vocabulary()

print("Tiny vocab:")
for index, token in enumerate(tiny_vocab):
    print(index, repr(token))

Tiny vocab:
0 ''
1 '[UNK]'
2 np.str_('story')
3 np.str_('movie')
4 np.str_('terrible')
5 np.str_('great')
6 np.str_('bad')
7 np.str_('amazing')


In [25]:
tiny_sentence = tf.constant(["great movie"])
tiny_ids = tiny_vectorizer(tiny_sentence)

In [26]:
tiny_sentence.numpy()[0].decode("utf-8")

'great movie'

In [27]:
tiny_ids.numpy()

array([[5, 3, 0, 0]])

In [28]:
tiny_ids.shape

TensorShape([1, 4])

### Step 6 -- Turn Token IDs Into Embedding Vectors

Now we can start creating the tiny embedding layer

In [29]:
tiny_embedding = tf.keras.layers.Embedding(
    input_dim=len(tiny_vocab),
    output_dim=3,
)
tiny_vectors = tiny_embedding(tiny_ids)

In [30]:
tiny_vectors.numpy()

array([[[ 0.0265666 , -0.04811701, -0.03076191],
        [ 0.01307596, -0.04135817, -0.00536425],
        [ 0.03581891,  0.01626286, -0.01412892],
        [ 0.03581891,  0.01626286, -0.01412892]]], dtype=float32)

In [31]:
tiny_vectors.shape

TensorShape([1, 4, 3])

### Step 7 -- Understand the Embedding Shape

###### The embedding output shape has three parts:
###### [batch_size, sequence_length, embedding_dim]
###### Each dimension tells a story about how the text is organized for the model

### Part2 - Load a Real Movie Review Dataset

In [32]:
train_data, test_data = tfds.load(
    name='imdb_reviews',
    split=['train[:10000]', 'test[:5000]'],
    as_supervised=True,
)

Dl Completed...: 0 url [00:00, ? url/s]
Dl Completed...:   0%|          | 0/1 [00:00<?, ? url/s]
Dl Size...: 0 MiB [00:00, ? MiB/s]WARNING:urllib3.connectionpool:Retrying (Retry(total=9, connect=None, read=None, redirect=None, status=None)) after connection broken by 'ConnectTimeoutError(<HTTPConnection(host='ai.stanford.edu', port=80) at 0x184dcfdacf0>, 'Connection to ai.stanford.edu timed out. (connect timeout=None)')': /~amaas/data/sentiment/aclImdb_v1.tar.gz
Dl Size...: 0 MiB [04:06, ? MiB/s]
Dl Completed...:   0%|          | 0/1 [04:06<?, ? url/s]


KeyboardInterrupt: 

In [33]:
dataset = load_dataset("stanfordnlp/imdb")

# Save it to a local folder
dataset.save_to_disk("./imdb_stanfordnlp")
print("Dataset saved to ./imdb_local")

Saving the dataset (1/1 shards): 100%|██████████| 50000/50000 [00:00<00:00, 369520.06 examples/s]

Dataset saved to ./imdb_local


In [58]:
# Load from local disk (no internet needed)
# dataset = load_from_disk("./imdb_local")

train_hf = dataset["train"].shuffle(seed=42).select(range(10000))
test_hf  = dataset["test"].shuffle(seed=42).select(range(5000))

train_data = tf.data.Dataset.from_tensor_slices((
        list(train_hf["text"]),
        list(train_hf["label"])
    ))
test_data  = tf.data.Dataset.from_tensor_slices((
        list(test_hf["text"]),
        list(test_hf["label"])
    ))

print(train_data)
print(test_data)

<_TensorSliceDataset element_spec=(TensorSpec(shape=(), dtype=tf.string, name=None), TensorSpec(shape=(), dtype=tf.int32, name=None))>
<_TensorSliceDataset element_spec=(TensorSpec(shape=(), dtype=tf.string, name=None), TensorSpec(shape=(), dtype=tf.int32, name=None))>


In [60]:
for review, label in train_data.take(2):
    print("review:")
    print(review.numpy().decode("utf-8")[:700], "...")
    print(label.numpy())
    print("-" * 80)

review:
There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. Fortier's plot are far more complicated... Fortier looks more like Prime Suspect, if we have to spot similarities... The main character is weak and weirdo, but have "clairvoyance". People like to compare, to judge, to evaluate. How about just enjoying? Funny thing too, people writing Fortier looks American but, on the other hand, arguing they prefer American series (!!!). Maybe it's the language, or the spirit, but I think this series is more English than American. By the way, the actors are real ...
1
--------------------------------------------------------------------------------
review:
This movie is a great. The plot is very true to the book which is a classic written by Mark Twain. The movie starts of with a scene where Hank sings a song with a bunch of kids called "when you st

In [61]:
max_tokens = 10000
sequence_length = 120

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode='int',
    output_sequence_length=sequence_length,
)

train_text_only = train_data.map(lambda text, label: text)

vectorizer.adapt(train_text_only.batch(1024))

vocab = vectorizer.get_vocabulary()

print("Vocabulary size: ", len(vocab))
print("First 20 vocabulary items: ")
print(vocab[:20])
print("-" * 80)

10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 135ms/step
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 135ms/step
Vocabulary size:  10000
First 20 vocabulary items: 
['', '[UNK]', np.str_('the'), np.str_('and'), np.str_('a'), np.str_('of'), np.str_('to'), np.str_('is'), np.str_('in'), np.str_('it'), np.str_('i'), np.str_('this'), np.str_('that'), np.str_('br'), np.str_('was'), np.str_('as'), np.str_('with'), np.str_('for'), np.str_('but'), np.str_('movie')]
--------------------------------------------------------------------------------
